# OpenDP polars integration to Lomas

In [67]:
# Lomas client setup
from lomas_client import Client

APP_URL = "http://localhost:80"
USER_NAME = "Dr. FSO"
DATASET_NAME = "FSO_INCOME_SYNTHETIC"
client = Client(url=APP_URL, user_name = USER_NAME, dataset_name = DATASET_NAME)

metadata = client.get_dataset_metadata()

In [94]:
import polars as pl
import opendp.prelude as dp
from typing import OrderedDict

dp.enable_features("contrib")

path = "../../server/data/datasets/income_synthetic_data.csv"
lf = pl.scan_csv(path)


def get_lf_domain(metadata):
    series_domains = []

    # Series domains
    for name, series_info in metadata["columns"].items():
        if "type" not in series_info:
            raise Exception("Missing type info in metadata") # TODO change exception
        
        # Note: this will use default types (eg. int32, etc.), do we want something else, or can we be more precise in metadata?
        # Note: bug here: https://github.com/opendp/opendp/blob/0797085a91bd16a48baf329956fd85a0f5a7ce38/python/src/opendp/typing.py#L242
        #       which means that arbitrary strings simply get returned..
        #       "datetime" is only smartnoiseSQL type that would fall in this category.
        # Note: what is the difference with get_atom?
        series_type = dp.RuntimeType.parse(series_info["type"])

        # Note: Same as using option_domain (at least how I understand it)
        series_nullable = True if "nullable" in series_info else False
        
        series_bounds = None
        if "lower" in series_info and "upper" in series_info:
            series_bounds = (series_info["lower"], series_info["upper"])
        
        series_domain = dp.series_domain(name, dp.atom_domain(T=series_type, nullable=series_nullable, bounds=series_bounds))
        series_domains.append(series_domain)

    lf_domain = dp.lazyframe_domain(series_domains)

    # Margins
    # TODO Check lenghts vs. keys for public info
    if "rows" in metadata:
        lf_domain = dp.with_margin(lf_domain, by=[], public_info="lengths", max_partition_length=metadata["rows"], max_num_partitions=1)
    
    for name, series_info in metadata["columns"].items():
        # TODO what about other info (max_partition_length, etc..)
        if "max_num_partitions" in series_info:
            lf_domain = dp.with_margin(lf_domain, by=[name], public_info="lengths", max_num_partitions=series_info["max_num_partitions"])
    
    return lf_domain


def get_lf_seed(metadata):
    schema = OrderedDict()
    for name, series_info in metadata["columns"].items():
        if "type" not in series_info:
            raise Exception("Missing type info in metadata") # TODO change exception
        
        match series_info["type"]:
            case "int":
                series_type = pl.datatypes.Int32 # TODO 64 or 32? depends on opendp's default
            case "float":
                series_type = pl.datatypes.Float64 # TODO same as above.
            case "string":
                series_type = pl.datatypes.String
            case "boolean":
                series_type = pl.datatypes.Bool
            case _:
                raise Exception(f"Type {series_info['type']} not supported by OpenDP.") # TODO change exception
        
        schema[name] = series_type
            
    return pl.DataFrame(None, schema, orient="row").lazy()
    

In [95]:
#metadata["columns"]["income"]["upper"] = 100000.0
#metadata["columns"]["income"]["lower"] = 1000.0

lf_domain = get_lf_domain(metadata)
seed = get_lf_seed(metadata)

In [96]:
# TODO why does one need to specify bounds if they already are in the margins
plan = seed.select(
    pl.col("income").dp.mean(bounds=(1000.0, 100000.0), scale=1.0)
)

# TODO: should user only select divergence type?
def opendp_measurement(plan, lf_domain):
    return dp.m.make_private_lazyframe(lf_domain, dp.symmetric_distance(), dp.max_divergence(T=float), plan)

m_lf = opendp_measurement(plan, lf_domain) # Could define global noise value

mean_income = m_lf(lf).collect()
mean_income

income
f64
7052.657818


In [99]:
plan.serialize()

'{"Projection":{"expr":[{"BinaryExpr":{"left":{"Function":{"input":[{"Agg":{"Sum":{"Function":{"input":[{"Column":"income"},{"Literal":{"Float64":1000.0}},{"Literal":{"Float64":100000.0}}],"function":{"Clip":{"has_min":true,"has_max":true}},"options":{"collect_groups":"ElementWise","fmt_str":"","input_wildcard_expansion":false,"returns_scalar":false,"cast_to_supertypes":false,"allow_rename":false,"pass_name_to_apply":false,"changes_length":false,"check_lengths":true,"allow_group_aware":true}}}}}],"function":{"FfiPlugin":{"lib":"/home/azureuser/work/venv-opendp0.10/lib/python3.11/site-packages/opendp/lib/opendp.abi3.so","symbol":"noise","kwargs":[128,5,149,38,0,0,0,0,0,0,0,125,148,40,140,5,115,99,97,108,101,148,71,63,240,0,0,0,0,0,0,140,12,100,105,115,116,114,105,98,117,116,105,111,110,148,78,117,46]}},"options":{"collect_groups":"ElementWise","fmt_str":"","input_wildcard_expansion":false,"returns_scalar":false,"cast_to_supertypes":false,"allow_rename":false,"pass_name_to_apply":false,"

In [97]:
from opendp_logger import enable_logging
from opendp.mod import enable_features
enable_features("contrib")
enable_logging()
m_lf.to_json()

ValueError: invoke `opendp_logger.enable_logging()` before constructing your measurement

1. get metadata
2. get lf_domain from metadata
3. get schema from metadata, or get seed from metadata directly?

4. make plan -> can define type of output measure (or not) and scale (or not)
6. make measurement out of plan (requires schema) -> defines type of output measure and potentially global scale

7. apply measurement to lazyframe
8. cost.

